# Tools & Function Calling

**WatSPEED Agentic AI prep — Week 1-2 - function calling**

Runs offline. Set `OPENAI_API_KEY` to swap the stub model for a real one.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() if (pathlib.Path.cwd() / 'agentkit.py').exists()
                      else pathlib.Path.cwd() / 'notebooks'))
from agentkit import *

## Why this is the first agentic concept

An "agent" is a loop around one primitive: **the model can ask your code to run a
function, and gets the result back**. Everything else — ReAct, LangGraph,
multi-agent — is control flow built on top of that one move.

The contract has three parts:

1. You describe your functions as **JSON Schema** and send them with the prompt.
2. The model replies with a *request*: `{"name": ..., "arguments": {...}}`. It never
   runs anything itself.
3. **You** validate, execute, and send the result back as a new message.

Step 3 is where your engineering lives. The model proposes; your code disposes.

### The "Why": Why Function Calling is a Paradigm Shift

> **The Legacy Friction:** In traditional SAS or Python data processing, if you need to extract structured data from unstructured text (like a survey comment), you have to write incredibly complex, fragile regular expressions or use `SCAN()` / `SUBSTR()`. When the text format slightly changes, your regex breaks silently.
>
> **The RAP Value Proposition:** Function Calling completely eliminates text parsing. You define a strict function schema (like a SAS `PROC FORMAT`), and the LLM guarantees it will output a cleanly formatted JSON object that perfectly matches your schema. You get structured, reliable data out of chaos without writing a single line of regex.


In [2]:
# A tool is just a function + a schema describing it.
tools = ToolBox()

@tools.tool("Cross-tabulate two survey columns and return a chi-square test",
            schema(row_var='string', col_var='string'))
def crosstab(row_var: str, col_var: str) -> dict:
    # Stand-in for a real pandas.crosstab + scipy.stats.chi2_contingency call.
    return {"row": row_var, "col": col_var, "chi2": 12.41, "dof": 3, "p_value": 0.0061}

@tools.tool("Return weighted mean of a numeric survey column",
            schema(column='string'))
def weighted_mean(column: str) -> dict:
    return {"column": column, "weighted_mean": 3.42, "n": 2041}

show("Tools the model will be told about", tools.names())

Tools the model will be told about:
  [
    "crosstab",
    "weighted_mean"
  ]


### What actually goes over the wire

This is the exact payload shape the OpenAI and Anthropic tool-calling APIs accept.
Read it closely — most agent bugs are schema bugs, and frameworks hide this from you.

In [3]:
show("JSON sent to the model", tools.schemas()[0])

JSON sent to the model:
  {
    "type": "function",
    "function": {
      "name": "crosstab",
      "description": "Cross-tabulate two survey columns and return a chi-square test",
      "parameters": {
        "type": "object",
        "properties": {
          "row_var": {
            "type": "string"
          },
          "col_var": {
            "type": "string"
          }
        },
        "required": [
          "row_var",
          "col_var"
        ]
      }
    }
  }


### The model's turn

The stub below is *scripted* so the notebook is reproducible. A real model would
produce the same structure from the prompt. Note it returns a **request**, not a result.

In [4]:
llm = get_llm([
    LLMResponse(tool_calls=[{"name": "crosstab",
                             "arguments": {"row_var": "age_group", "col_var": "trusts_ai"}}]),
    LLMResponse(content="Age group and AI trust are associated (chi2=12.41, p=0.006)."),
])

messages = [{"role": "user", "content": "Is AI trust related to age group?"}]
first = llm.chat(messages, tools=tools.schemas())

show("Model asked to call", first.tool_calls)
print("Did it answer directly?", first.content)

Using stub-llm (deterministic, offline) - no OPENAI_API_KEY found, so results are scripted.
Model asked to call:
  [
    {
      "name": "crosstab",
      "arguments": {
        "row_var": "age_group",
        "col_var": "trusts_ai"
      }
    }
  ]
Did it answer directly? None


### Closing the loop

You run the function, append the result as a `tool` message, and call the model again.
That append-and-recall is the whole mechanism.

In [5]:
call = first.tool_calls[0]
result = tools.call(call["name"], call["arguments"])       # <-- your code runs, not the model's
show("Tool returned", result)

messages += [
    {"role": "assistant", "tool_calls": first.tool_calls},
    {"role": "tool", "name": call["name"], "content": str(result)},
]
final = llm.chat(messages, tools=tools.schemas())
print("\nFinal answer:", final.content)

Tool returned:
  {
    "row": "age_group",
    "col": "trusts_ai",
    "chi2": 12.41,
    "dof": 3,
    "p_value": 0.0061
  }

Final answer: Age group and AI trust are associated (chi2=12.41, p=0.006).


### Validate before you execute

The model can emit arguments that don't match your schema. Never call a function on
unvalidated model output — this is the single most common agent security hole.
Pydantic (which the course expects you to know) gives you the check in three lines.

### The Deterministic Cage: Pydantic as the Bouncer

> **The Fear of Non-Determinism:** If LLMs are probabilistic (they hallucinate), how can we trust them to run our critical data pipelines? If the LLM decides to pass the string `"twenty"` instead of the integer `20` to our model, the Python script will crash.
>
> **The Solution:** We do not trust the LLM. We build a 'Deterministic Cage' around it. Pydantic acts as an aggressive bouncer at the door of your function. If the LLM tries to pass bad data, Pydantic intercepts the JSON packet, blocks it from hitting your actual Python code, and sends a strict error back to the LLM (`"Expected Integer, got String"`). The LLM is forced to self-correct.


In [6]:
from pydantic import BaseModel, ValidationError, field_validator

ALLOWED = {"age_group", "trusts_ai", "education", "region"}

class CrosstabArgs(BaseModel):
    row_var: str
    col_var: str

    @field_validator("row_var", "col_var")
    @classmethod
    def must_be_known_column(cls, v: str) -> str:
        if v not in ALLOWED:
            raise ValueError(f"unknown column {v!r}; allowed: {sorted(ALLOWED)}")
        return v

banner("Good arguments")
show("parsed", CrosstabArgs(**call["arguments"]).model_dump())

banner("Hallucinated column - rejected before it reaches your data")
try:
    CrosstabArgs(row_var="age_group", col_var="respondent_ssn")
except ValidationError as e:
    print(e.errors()[0]["msg"])


Good arguments
parsed:
  {
    "row_var": "age_group",
    "col_var": "trusts_ai"
  }

Hallucinated column - rejected before it reaches your data
Value error, unknown column 'respondent_ssn'; allowed: ['age_group', 'education', 'region', 'trusts_ai']


### The same thing in LangChain

The course teaches LangChain. This is the identical idea in its syntax — the cell is
guarded so it prints the equivalent code if the package isn't installed yet.

```python
from langchain_core.tools import tool

@tool
def crosstab(row_var: str, col_var: str) -> dict:
    """Cross-tabulate two survey columns and return a chi-square test."""
    ...

llm_with_tools = ChatOpenAI(model="gpt-4o-mini").bind_tools([crosstab])
```

LangChain reads the type hints and docstring to build the schema you just wrote by
hand. Knowing what it generates is what lets you debug it.

In [7]:
try:
    from langchain_core.tools import tool

    @tool
    def crosstab_lc(row_var: str, col_var: str) -> dict:
        """Cross-tabulate two survey columns and return a chi-square test."""
        return {"row": row_var, "col": col_var}

    show("LangChain generated this schema", crosstab_lc.args_schema.model_json_schema())
except ImportError:
    print("langchain-core not installed - see requirements.txt.")
    print("Compare with the hand-written schema above; they carry the same information.")

langchain-core not installed - see requirements.txt.
Compare with the hand-written schema above; they carry the same information.


### Guided Exercise: Wrapping PROC MEANS (groupby) as an AI Tool

In SAS, calculating summary statistics by category is done via `PROC MEANS CLASS=category_var`. Let's build a Python equivalent using `pandas.groupby()` and wrap it in a `@tool` decorator so an LLM can use it autonomously.

In [ ]:
from pydantic import BaseModel, Field

# 1. Define the Strict Input Schema (The Menu)
class GroupByInput(BaseModel):
    category_col: str = Field(..., description="The column to group by (e.g., 'Education_Level')")
    target_col: str = Field(..., description="The numeric column to average (e.g., 'Perceived_AI_Risk')")

# 2. Define the Python logic (The Kitchen)
# Assume `clean_ai` is our survey dataframe from earlier
def calculate_group_means(args: GroupByInput) -> str:
    try:
        # Python's equivalent to PROC MEANS CLASS=
        means = clean_ai.groupby(args.category_col)[args.target_col].mean().reset_index()
        return means.to_string(index=False)
    except Exception as e:
        return f"Error calculating means: {str(e)}"

print("Tool defined! An LLM can now autonomously compute group means without writing a single line of SAS.")